# ALMASim metadata query walkthrough

This notebook demonstrates how to:

1. query ALMA metadata directly from TAP
2. save the normalized metadata table as CSV
3. resolve product-level access URLs for selected `member_ous_uid` rows
4. save those product rows as CSV for later download workflows


In [ ]:
from pathlib import Path

from almasim.services.metadata.tap import (
    InclusionFilters,
    query_metadata_by_science,
    query_products,
)

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "almasim").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
metadata_csv = output_dir / "metadata_query_results.csv"
products_csv = output_dir / "metadata_products.csv"


In [ ]:
include = InclusionFilters(
    science_keyword=["Galaxies"],
    band=[6],
)
metadata = query_metadata_by_science(include=include).head(25).reset_index(drop=True)
metadata.to_csv(metadata_csv, index=False)

print(f"Saved metadata CSV: {metadata_csv}")
metadata[[col for col in ["ALMA_source_name", "Band", "Freq", "member_ous_uid"] if col in metadata.columns]].head()


In [ ]:
member_uids = metadata["member_ous_uid"].dropna().astype(str).head(1).tolist()
products = query_products(member_uids)
products.to_csv(products_csv, index=False)

print(f"Saved products CSV: {products_csv}")
products.head()


In [ ]:
print("Next step: pass metadata_csv into the staged pipeline example, or use the saved access_url rows in a separate download workflow.")
